In [60]:
using LinearAlgebra
using Plots
using DataStructures

## Rotational and Radial Kinetic Energy

$$\alpha,\alpha^\prime \in \{1, \dots N\} \space | \space \beta,\beta^\prime \in [0, 2 \pi]$$

Off Diagonal

$$T_{r \alpha \alpha^{\prime}} = \left(\frac{\hbar^2}{2\mu \Delta r^2 }\right) \cdot \left (-1\right)^{\left (\alpha - \alpha^{\prime} \right)} \cdot \left(\frac{\pi^2}{3} - \frac{1}{2\alpha^2}\right) $$

$${T}_{\phi \beta \beta^{\prime}} = \left(\frac{\hbar^2}{2\mu}\right) \cdot \left (-1\right)^{\left (\beta - \beta^{\prime} \right)} \cdot \left(\frac{N(N+1)}{3} \right)$$

On Diagonal 

$$T_{r \alpha \alpha^{\prime}} = \left(\frac{\hbar^2}{2\mu \Delta r^2}\right) \cdot \left(\frac{2}{(\alpha-\alpha^{\prime})^2} - \frac{2}{(\alpha+\alpha^{\prime})^2} \right)$$

$${T}_{\phi \beta \beta^{\prime}} = \left(\frac{\hbar^2}{2\mu}\right) \cdot \frac{cos\left(\frac{\pi(\beta-\beta)}{2N+1}\right)}{2sin\left(\frac{\pi(\beta-\beta)}{2N+1}\right)^2}$$



## DVR functions

In [61]:
function dvr_rotational(N, mu)
    h_bar = 1
    phi_grid = zeros((N))
    T = zeros((N, N))
    for a=1:N
        phi_grid[a] = (2*pi*(a-1)/(N))
        for ap=1:N
            if a==ap # On-diagonal
                T[a,ap] = h_bar^2 / 2*mu * (-1)^(a-ap) * (N*(N+1)/3)
            else # Off-Diagonal
                T[a,ap] = h_bar^2 / 2*mu * (-1)^(a-ap) * (cos(pi*(a-ap)/(2*N+1)))/(2*sin(pi*(a-ap)/(2*N+1))^2)
            
            end
        end
    end
    return T, phi_grid
end

function dvr_radial(N, r_min, r_max, mu)
    h_bar = 1
    r_grid = zeros((N))
    T = zeros((N, N))
    d_r = (r_max-r_min)/(N-1)
    for i=1:N
        r_grid[i] = d_r*(i-1) + r_min
    end
    for b=1:N
        for bp=1:N
            if b==bp # On-diagonal 
                T[b,bp] = h_bar^2 / (2*mu*d_r^2) * (-1)^(b-bp) * (pi^2/3 - 1/(2*b^2))
            else # Off-diagonal
                T[b,bp] = h_bar^2 / (2*mu*d_r^2) * (-1)^(b-bp) * (2/(b-bp)^2 - 2/(b+bp)^2)
            end
        end
    end
    return T, r_grid
end

dvr_radial (generic function with 1 method)

## Harmonic and Dipole-Dipole Potential

In $(r, \phi)$ basis

$$V_{\alpha} = \frac{1}{2} k \left(r_{\alpha} - r_e \right)^2$$

$$\hat{V}_{1,2} = \frac{\left(\frac{\mu}{r_e} \right)^2}{4 \pi \epsilon_o R^3} \cdot \hat{r}_1 \hat{r}_2 \cdot \left(sin (\hat{\phi}_1) sin (\hat{\phi}_2) - 2 cos (\hat{\phi}_1) cos (\hat{\phi}_2) \right)$$

In [62]:
function V_oscilation(N, k, r_e, r_grid)

    V = zeros((N))

    for i=1:N
        V[i] = 0.5*k * (r_grid[i] - r_e)^2
    end
    return V
end

function V_neighbours(N_r, N_phi, mu, r_grid, phi_grid, r_e, R_3)
    epsilon_o = 1
    size = N_r*N_phi
    V = zeros((size, size))
    i = 0
    C = ((mu/r_e)^2)/(4*pi*epsilon_o*R_3^3)
    C = 0
    for a1=1:N_r
        for a2=1:N_r
            i+=1
            j=0
            temp = C * r_grid[a1]*r_grid[a2]
            for b1=1:N_phi
                for b2=1:N_phi
                    j+=1
                    V[i,j] = temp * sin(phi_grid[b1])*sin(phi_grid[b2]) - 2*cos(phi_grid[b1])*cos(phi_grid[b2])
                end
            end
        end
    end
    return V
end

V_neighbours (generic function with 1 method)

## Constants for HF molecule

In [63]:
amu_to_au = 1/0.00054858 #au
ang_to_bohr = 1/0.529177 #1/bohr radius

wavenumber_to_Hartrees = 0.00000455633

m_H = 1.008 #in amu
m_F = 18.998 #in amu

r_e = 0.9168 * ang_to_bohr #in Angstrom

omega_e = 4138.385 #Hartrees

m_H *= amu_to_au
m_F *= amu_to_au

mu = (m_H*m_F)/(m_H+m_F) #reduced mass

omega_e *= wavenumber_to_Hartrees #h_bar = 1 (in au)

k = omega_e^2 * mu

Alpha = (mu*omega_e)/2 # h_bar not included as h_bar = 1

16.450694885570016

$$e^{(-\frac{\mu \omega r^2}{2 \hbar})}$$

For gaussian Distribution of vibrational states

$$\alpha (r-r_e)^2 = order$$

$$r = r_e \pm \sqrt{\frac{order}{\alpha}}

In [64]:
order = 5

r_min = r_e - sqrt(order/Alpha)
r_max = r_e + sqrt(order/Alpha)

2.2838078038699123

In [65]:
N_r = 24
N_phi = 24

mmax = 10

R = 10 #Distance between rotors

10

In [66]:
T_r, r_grid = dvr_radial(N_r, r_min, r_max, mu)

T_phi, phi_grid = dvr_rotational(N_phi, mu)

V_r = V_oscilation(N_r, k, r_e, r_grid)

24-element Vector{Float64}:
 0.09427923863524999
 0.07859573579989648
 0.06433800594957512
 0.05150604908428588
 0.04009986520402885
 0.03011945430880388
 0.021564816398611058
 0.014435951473450372
 0.008732859533321825
 0.004455540578225416
 0.0016039946081611459
 0.000178221623129014
 0.0001782216231290206
 0.0016039946081611656
 0.004455540578225433
 0.008732859533321848
 0.014435951473450401
 0.021564816398611093
 0.030119454308803965
 0.04009986520402885
 0.051506049084285944
 0.06433800594957519
 0.07859573579989655
 0.09427923863525009

## 1-body Hamiltonian, in the $(\alpha, m)$ basis

In [67]:
function Hamiltonian_1body(N_r, T_r, V_r, mu, m)
    H = zeros((N_r, N_r))
    for a=1:N_r
        H[a,a] = V_r[a] + 1/(2*mu*r_grid[a]^2) * m^2
        for ap=1:N_r
            H[a,ap] += T_r[a,ap]
        end
    end
    return H
end

Hamiltonian_1body (generic function with 1 method)

A $2 \cdot (2 \space mmax + 1) \times 2 \cdot (2 \space mmax + 1)$ will be constructed for each of the following

$$ \langle n_m | r \cos \phi | n_m' \rangle = \sum_{\alpha, m} \langle n | \alpha ; m \rangle r_{\alpha} \cdot \frac{1}{2} (\langle \alpha ; m+1 | n' \rangle + \langle \alpha ; m+1 | n' \rangle ) $$

$$ \langle n_m | r \sin \phi | n_m' \rangle = \sum_{\alpha, m} \langle n | \alpha ; m \rangle r_{\alpha} \cdot \frac{1}{2} (\langle \alpha ; m+1 | n' \rangle - \langle \alpha ; m-1 | n' \rangle ) $$

In [68]:
mmax = 10

N_states = 2
N_m = 2*mmax + 1

N_max = N_states*N_m

cos_matrix = zeros(ComplexF64, N_max, N_max)
sin_matrix = zeros(ComplexF64, N_max, N_max)

h_m = Hamiltonian_1body(N_r, T_r, V_r, mu, -mmax)
evec_m = eigvecs(h_m)[:,1:N_states]

h_m_1up = Hamiltonian_1body(N_r, T_r, V_r, mu, -mmax+1)
evec_m_1up = eigvecs(h_m_1up)[:,1:N_states]

evec_m_1down = 0

for m=-mmax:mmax
    m_idx = m + mmax

    for n=1:N_states
        for np=1:N_states
            delta_m_up = 0 + 0im
            delta_m_down = 0 + 0im

            # Sum over all alpha values
            for a=1:N_r
                r = r_grid[a]
                val_m = evec_m[a,n]

                # Boundary edge cases
                # if m is not mmax
                if m < mmax
                    delta_m_up += val_m * r * evec_m_1up[a, np]
                end

                # If m is not -mmax
                if m > -mmax
                    delta_m_down += val_m * r * evec_m_1down[a, np]
                end
            end

            row_idx = n+N_states * m_idx  #alpha index

            # m index
            if m < mmax
                col_idx_up = np + N_states*(m_idx+1)
            end
            if m > -mmax
                col_idx_down = np + N_states*(m_idx-1)
            end
            
            if m == -mmax
                cos_matrix[row_idx, col_idx_up] = 1/2 * delta_m_up
                sin_matrix[row_idx, col_idx_up] = 1/2im * delta_m_up
            elseif m == mmax
                cos_matrix[row_idx, col_idx_down] = 1/2 * delta_m_down
                sin_matrix[row_idx, col_idx_down] = 1/2im * -delta_m_down
            else
                cos_matrix[row_idx, col_idx_up] = 1/2 * delta_m_up
                cos_matrix[row_idx, col_idx_down] = 1/2 * delta_m_down
                sin_matrix[row_idx, col_idx_up] = 1/2im * delta_m_up
                sin_matrix[row_idx, col_idx_down] = 1/2im * -delta_m_down
            end
        end
    end

    evec_m_1down = evec_m
    evec_m = evec_m_1up

    if m < mmax
        h_m_1up = Hamiltonian_1body(N_r, T_r, V_r, mu, m+1)
        evec_m_1up = eigvecs(h_m_1up)[:, 1:N_states]
    end
end

# Dipole-Dipole Interaction calculation (2-body)

$$D = \frac{1}{4 \pi \epsilon_o R^3} \frac{1}{N_r} \sum_{\alpha=1}^{N_r} \left(\frac{\mu}{r_\alpha}\right)^2$$

$$V_{12} = D \cdot (r_1 \sin(\phi_1) \cdot r_2 \sin(\phi_2) - 2 \cdot r_1 \cos(\phi_1) \cdot r_2 \cos(\phi_2))$$

As matricies

$$V_{12} = D(S_1 \otimes S_2 - 2 \cdot C_1 \otimes C_2)$$

Given

$$\langle n_m | r \sin \phi | n_m' \rangle = S, \langle n_m | r \cos \phi | n_m' \rangle = C$$

Overall Hamiltonian

$$H = h_1 + h_2 + V_{12}$$

Order = 5, from gaussian distribution of vibrational states (seen above)

$$\frac{1}{N_r} \sum_{\alpha=1}^{N_r} \left(\frac{\mu}{r_\alpha}\right)^2 = 0.19342$$

$$\left(\frac{\mu}{r_e}\right)^2 = 0.172133$$

In [ ]:
dipole_sum = 0

for i=1:N_r
    dipole_sum += (dipole_moment/r_grid[i])^2
end

dipole = (dipole_moment/r_e)^2

println(dipole_sum/N_r)
println(dipole)

0.19341569795064695
0.17213342977670262


In [ ]:
dipole_moment = 0.718797 #au, from 1.827 Debye (D), NIST database
epsilon_0 = 1
R = 10 #au

dipole_sum = 0

for i=1:N_r
    dipole_sum += (dipole_moment/r_grid[i])^2
end

#dipole_per_r = (dipole_moment/r_e)^2 #

D = 1/(4 * pi * epsilon_0 * R^3) * 1/N_r * dipole_sum

1.5391532200207215e-5

In [71]:
V_12 = D * (sin_matrix * sin_matrix - 2 * cos_matrix * cos_matrix)

V_12

42×42 Matrix{ComplexF64}:
 -1.18448e-5+0.0im  -1.64694e-6+0.0im  …          0.0+0.0im
 -1.64694e-6+0.0im  -1.18044e-5+0.0im             0.0+0.0im
         0.0+0.0im          0.0+0.0im             0.0+0.0im
         0.0+0.0im          0.0+0.0im             0.0+0.0im
 -3.54652e-5+0.0im  -4.46877e-6+0.0im             0.0+0.0im
  5.41229e-6+0.0im   3.54153e-5+0.0im  …          0.0+0.0im
         0.0+0.0im          0.0+0.0im             0.0+0.0im
         0.0+0.0im          0.0+0.0im             0.0+0.0im
         0.0+0.0im          0.0+0.0im             0.0+0.0im
         0.0+0.0im          0.0+0.0im             0.0+0.0im
         0.0+0.0im          0.0+0.0im  …          0.0+0.0im
         0.0+0.0im          0.0+0.0im             0.0+0.0im
         0.0+0.0im          0.0+0.0im             0.0+0.0im
            ⋮                          ⋱  
         0.0+0.0im          0.0+0.0im  …          0.0+0.0im
         0.0+0.0im          0.0+0.0im             0.0+0.0im
         0.0+0.0im          0.0